# Shrimp Disease YOLO Auto-Augmentation Benchmark

Trains the original YOLO classification variants with TrivialAugmentWide, RandAugment, and AugMix. RandAugment and AugMix use Ultralytics' built-in classification `auto_augment` modes. TrivialAugmentWide is implemented as a small custom PIL-based policy and used to create one offline augmented replacement for each original training image. Validation and test splits remain unaugmented.

In [ ]:
print("Running Kaggle YOLO auto-augmentation benchmark. Input data is read from /kaggle/input; outputs are written to /kaggle/working.")

## 1. Install Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "ultralytics>=8.3.0",
]


def import_name_for(package_spec: str) -> str:
    package = package_spec.split(">=")[0].split("==")[0].split("<")[0]
    return {"ultralytics": "ultralytics"}.get(package, package.replace("-", "_"))


missing = [pkg for pkg in packages if importlib.util.find_spec(import_name_for(pkg)) is None]
if missing:
    print("Installing missing dependencies:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("All training dependencies are already available.")


## 2. Configuration and Fixed Split

In [ ]:
import gc
import random
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageEnhance, ImageOps
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 224
EPOCHS = 30
PATIENCE = 5
YOLO_BATCH_SIZE = 32
ENABLE_GPU_LOGGING = True
GPU_LOG_EVERY_N_EPOCHS = 1

DATA_DIR = Path("/kaggle/input/datasets/uynnhy/processed-images/processed_images")
OUTPUT_DIR = Path("/kaggle/working/yolo_autoaugment_model_comparison")
YOLO_DATA_ROOT = OUTPUT_DIR / "yolo_datasets"
YOLO_RUNS_DIR = OUTPUT_DIR / "yolo_runs"
REPORT_DIR = OUTPUT_DIR / "reports"

for directory in [YOLO_DATA_ROOT, YOLO_RUNS_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = len(CLASS_DIRS)

YOLO_MODELS = ["yolo26n-cls", "yolo26m-cls", "yolo11n-cls", "yolo11m-cls"]
ALL_MODELS = YOLO_MODELS.copy()

AUTO_AUGMENT_POLICIES = {
    "trivial_aug": "TrivialAugmentWide",
    "randaug": "RandAugment",
    "augmix": "AugMix",
}

YOLO_AUTO_AUGMENT_BY_POLICY = {
    "trivial_aug": None,
    "randaug": "randaugment",
    "augmix": "augmix",
}


def reset_random_state(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sanitize_name(name: str) -> str:
    return name.replace("/", "_").replace(" ", "_").replace(".", "_")


def count_params(model) -> float:
    return sum(param.numel() for param in model.parameters()) / 1e6


def gpu_status_text() -> str:
    if not torch.cuda.is_available():
        return "CUDA unavailable"

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    max_allocated = torch.cuda.max_memory_allocated() / 1024**2
    text = f"torch CUDA memory allocated/reserved/max: {allocated:.1f}/{reserved:.1f}/{max_allocated:.1f} MB"

    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )
        if completed.returncode == 0:
            first_gpu = completed.stdout.strip().splitlines()[0]
            util, mem_used, mem_total, power, temp = [part.strip() for part in first_gpu.split(",")[:5]]
            text += f" | nvidia-smi util={util}% mem={mem_used}/{mem_total} MB power={power} W temp={temp} C"
        else:
            text += f" | nvidia-smi failed: {completed.stderr.strip()[-200:]}"
    except Exception as exc:
        text += f" | nvidia-smi unavailable: {type(exc).__name__}: {exc}"

    return text


def print_gpu_status(label: str):
    if ENABLE_GPU_LOGGING:
        print(f"[GPU] {label}: {gpu_status_text()}")


print_gpu_status("startup")


In [ ]:
def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f"Warning: missing class folder: {folder}")
            continue
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                rows.append({"path": str(path), "class_dir": class_dir, "label": CLASS_TO_IDX[class_dir]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No images found under {data_dir}. Run preprocessing from the baseline first.")
    return frame


df = discover_processed_images(DATA_DIR)
print(f"Loaded {len(df)} processed images from {DATA_DIR}")
display(df["class_dir"].value_counts().reindex(CLASS_DIRS).rename("count").to_frame())

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df["label"],
    random_state=SEED,
    shuffle=True,
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{split_name}: {len(split_df)} images")
    print(split_df["class_dir"].value_counts().reindex(CLASS_DIRS).to_dict())

assert set(train_df["path"]).isdisjoint(set(val_df["path"]))
assert set(train_df["path"]).isdisjoint(set(test_df["path"]))
assert set(val_df["path"]).isdisjoint(set(test_df["path"]))
print("Image-level split overlap check passed.")


## 3. YOLO Datasets and Custom TrivialAugmentWide

RandAugment and AugMix use the original training split with Ultralytics built-in `auto_augment`. TrivialAugmentWide uses one custom offline augmented replacement per training image.

In [ ]:
TRIVIAL_AUGMENT_BINS = 31
TRIVIAL_AUGMENT_OPS = [
    "identity",
    "autocontrast",
    "equalize",
    "rotate",
    "solarize",
    "color",
    "contrast",
    "brightness",
    "sharpness",
    "shear_x",
    "shear_y",
    "translate_x",
    "translate_y",
]


def magnitude_value(max_value: float, signed: bool = False):
    magnitude = random.randint(0, TRIVIAL_AUGMENT_BINS - 1) / (TRIVIAL_AUGMENT_BINS - 1)
    value = magnitude * max_value
    if signed and random.random() < 0.5:
        value *= -1
    return value


def apply_custom_trivial_augment(image: Image.Image) -> Image.Image:
    op = random.choice(TRIVIAL_AUGMENT_OPS)
    if op == "identity":
        return image
    if op == "autocontrast":
        return ImageOps.autocontrast(image)
    if op == "equalize":
        return ImageOps.equalize(image)
    if op == "rotate":
        degrees = magnitude_value(30.0, signed=True)
        return image.rotate(degrees, resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    if op == "solarize":
        threshold = int(256 - magnitude_value(256.0))
        return ImageOps.solarize(image, threshold=max(0, min(256, threshold)))
    if op == "color":
        factor = 1.0 + magnitude_value(0.9, signed=True)
        return ImageEnhance.Color(image).enhance(max(0.1, factor))
    if op == "contrast":
        factor = 1.0 + magnitude_value(0.9, signed=True)
        return ImageEnhance.Contrast(image).enhance(max(0.1, factor))
    if op == "brightness":
        factor = 1.0 + magnitude_value(0.9, signed=True)
        return ImageEnhance.Brightness(image).enhance(max(0.1, factor))
    if op == "sharpness":
        factor = 1.0 + magnitude_value(0.9, signed=True)
        return ImageEnhance.Sharpness(image).enhance(max(0.1, factor))
    if op == "shear_x":
        shear = magnitude_value(0.3, signed=True)
        return image.transform(image.size, Image.Transform.AFFINE, (1, shear, 0, 0, 1, 0), resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    if op == "shear_y":
        shear = magnitude_value(0.3, signed=True)
        return image.transform(image.size, Image.Transform.AFFINE, (1, 0, 0, shear, 1, 0), resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    if op == "translate_x":
        offset = int(round(magnitude_value(0.45, signed=True) * image.size[0]))
        return image.transform(image.size, Image.Transform.AFFINE, (1, 0, offset, 0, 1, 0), resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    if op == "translate_y":
        offset = int(round(magnitude_value(0.45, signed=True) * image.size[1]))
        return image.transform(image.size, Image.Transform.AFFINE, (1, 0, 0, 0, 1, offset), resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128))
    raise ValueError(f"Unsupported TrivialAugment op: {op}")


def resize_crop_for_yolo(image: Image.Image) -> Image.Image:
    width, height = image.size
    scale = random.uniform(0.82, 1.0)
    aspect = random.uniform(0.90, 1.10)
    crop_area = width * height * scale
    crop_w = int(round((crop_area * aspect) ** 0.5))
    crop_h = int(round((crop_area / aspect) ** 0.5))
    crop_w = min(width, max(1, crop_w))
    crop_h = min(height, max(1, crop_h))
    left = random.randint(0, max(0, width - crop_w))
    top = random.randint(0, max(0, height - crop_h))
    image = image.crop((left, top, left + crop_w, top + crop_h))
    return image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BICUBIC)


def apply_trivial_aug_train_transform(image: Image.Image) -> Image.Image:
    return apply_custom_trivial_augment(resize_crop_for_yolo(image))


def save_image_for_yolo(image: Image.Image, dst: Path):
    image.convert("RGB").save(dst, quality=95)


def unique_image_name(src: Path, row_idx: int, suffix: str = ".jpg") -> str:
    return f"{src.stem}_{row_idx:06d}{suffix}"


def prepare_yolo_dataset_for_augmentation(augmentation_key: str) -> Path:
    augmentation_name = AUTO_AUGMENT_POLICIES[augmentation_key]
    dataset_dir = YOLO_DATA_ROOT / augmentation_key
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    for split_name in ["train", "val", "test"]:
        for class_dir in CLASS_DIRS:
            (dataset_dir / split_name / class_dir).mkdir(parents=True, exist_ok=True)

    reset_random_state(SEED)
    if augmentation_key == "trivial_aug":
        print(f"Creating custom offline {augmentation_name} training split at {dataset_dir / 'train'}")
    else:
        print(f"Copying original training split for built-in YOLO {augmentation_name} at {dataset_dir / 'train'}")

    for row_idx, (_, row) in enumerate(tqdm(train_df.iterrows(), total=len(train_df), desc=f"Build {augmentation_name} train")):
        src = Path(row["path"])
        if augmentation_key == "trivial_aug":
            dst = dataset_dir / "train" / row["class_dir"] / unique_image_name(src, row_idx)
            image = Image.open(src).convert("RGB")
            save_image_for_yolo(apply_trivial_aug_train_transform(image), dst)
        else:
            dst = dataset_dir / "train" / row["class_dir"] / unique_image_name(src, row_idx, src.suffix.lower() or ".jpg")
            shutil.copy2(src, dst)

    for split_name, split_df in [("val", val_df), ("test", test_df)]:
        for row_idx, (_, row) in enumerate(tqdm(split_df.iterrows(), total=len(split_df), desc=f"Copy {split_name}")):
            src = Path(row["path"])
            dst = dataset_dir / split_name / row["class_dir"] / unique_image_name(src, row_idx, src.suffix.lower() or ".jpg")
            shutil.copy2(src, dst)

    print(f"Prepared YOLO dataset for {augmentation_name}: {dataset_dir}")
    return dataset_dir


## 4. YOLO Training and Evaluation

In [ ]:
def yolo_device_arg():
    return 0 if torch.cuda.is_available() else "cpu"


def evaluate_yolo_model(yolo_model, eval_df: pd.DataFrame, timed=False):
    names = yolo_model.names
    name_to_idx = {value: int(key) for key, value in names.items()}
    source_paths = eval_df["path"].tolist()

    if timed:
        _ = yolo_model.predict(source=source_paths[:1], imgsz=IMG_SIZE, device=yolo_device_arg(), verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
    else:
        start = None

    preds = yolo_model.predict(
        source=source_paths,
        imgsz=IMG_SIZE,
        batch=YOLO_BATCH_SIZE,
        device=yolo_device_arg(),
        verbose=False,
    )

    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None

    y_true = [name_to_idx[class_dir] for class_dir in eval_df["class_dir"].tolist()]
    y_pred = [int(result.probs.top1) for result in preds]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "elapsed": elapsed,
        "labels": y_true,
        "preds": y_pred,
    }


YOLO_AUGMENTATION_BASE_ARGS = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "erasing": 0.0,
    "crop_fraction": 1.0,
}


def yolo_augmentation_args(augmentation_key: str) -> dict:
    args = YOLO_AUGMENTATION_BASE_ARGS.copy()
    args["auto_augment"] = YOLO_AUTO_AUGMENT_BY_POLICY[augmentation_key]
    return args


def train_yolo_model(model_name: str, augmentation_key: str, dataset_dir: Path) -> dict:
    augmentation_name = AUTO_AUGMENT_POLICIES[augmentation_key]
    print("\n" + "=" * 90)
    print(f"Training YOLO classifier: {model_name} | Augmentation: {augmentation_name}")
    print("=" * 90)

    weights_name = f"{model_name}.pt"
    run_name = f"{augmentation_key}_{sanitize_name(model_name)}"
    train_start = time.time()
    yolo = YOLO(weights_name)
    print_gpu_status(f"{augmentation_name} / {model_name} before YOLO train")
    yolo.train(
        data=str(dataset_dir),
        task="classify",
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=YOLO_BATCH_SIZE,
        patience=PATIENCE,
        seed=SEED,
        project=str(YOLO_RUNS_DIR),
        name=run_name,
        exist_ok=True,
        device=yolo_device_arg(),
        verbose=True,
        **yolo_augmentation_args(augmentation_key),
    )
    print_gpu_status(f"{augmentation_name} / {model_name} after YOLO train")
    train_time = time.time() - train_start

    best_path = YOLO_RUNS_DIR / run_name / "weights" / "best.pt"
    if not best_path.exists():
        candidates = sorted((YOLO_RUNS_DIR / run_name).glob("**/best.pt"))
        if not candidates:
            raise FileNotFoundError(f"Could not locate YOLO best checkpoint for {model_name} / {augmentation_name}")
        best_path = candidates[-1]

    best_yolo = YOLO(str(best_path))
    print_gpu_status(f"{augmentation_name} / {model_name} before YOLO eval")
    val_metrics = evaluate_yolo_model(best_yolo, val_df, timed=False)
    test_metrics = evaluate_yolo_model(best_yolo, test_df, timed=True)
    print_gpu_status(f"{augmentation_name} / {model_name} after YOLO eval")
    inf_time = test_metrics["elapsed"]
    params_m = count_params(best_yolo.model)

    result = {
        "Augmentation": augmentation_name,
        "Augmentation Key": augmentation_key,
        "Model": model_name,
        "Backend Name": weights_name,
        "Parameters (M)": round(params_m, 2),
        "Training Time (s)": round(train_time, 1),
        "Val F1-Score": round(val_metrics["macro_f1"], 4),
        "Test Accuracy": round(test_metrics["accuracy"], 4),
        "Test F1-Score": round(test_metrics["macro_f1"], 4),
        "Cohen Kappa": round(test_metrics["cohen_kappa"], 4),
        "Inference Time (s)": round(inf_time, 2),
        "FPS": round(len(test_df) / inf_time, 1),
        "Latency (ms)": round((inf_time / len(test_df)) * 1000, 2),
        "Checkpoint Path": str(best_path),
        "Export Format": "",
        "Export Path": "",
        "Export Note": f"YOLO auto_augment={YOLO_AUTO_AUGMENT_BY_POLICY[augmentation_key]}; other augmentation knobs disabled. TrivialAugmentWide uses custom offline replacement images.",
    }

    del yolo, best_yolo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result


## 5. Sequential YOLO Auto-Augmentation Loop

Partial CSV/JSON files are saved after every model.

In [ ]:
comparison_results = []

for augmentation_key, augmentation_name in AUTO_AUGMENT_POLICIES.items():
    print("\n" + "#" * 90)
    print(f"Starting YOLO augmentation benchmark: {augmentation_name}")
    print("#" * 90)
    dataset_dir = prepare_yolo_dataset_for_augmentation(augmentation_key)

    for model_name in ALL_MODELS:
        try:
            reset_random_state(SEED)
            result = train_yolo_model(model_name, augmentation_key, dataset_dir)
            comparison_results.append(result)
            print("Recorded result:")
            display(pd.DataFrame([result]))
        except Exception as exc:
            print(f"ERROR while running {augmentation_name} / {model_name}: {type(exc).__name__}: {exc}")
            comparison_results.append(
                {
                    "Augmentation": augmentation_name,
                    "Augmentation Key": augmentation_key,
                    "Model": model_name,
                    "Backend Name": "",
                    "Parameters (M)": np.nan,
                    "Training Time (s)": np.nan,
                    "Val F1-Score": np.nan,
                    "Test Accuracy": np.nan,
                    "Test F1-Score": np.nan,
                    "Cohen Kappa": np.nan,
                    "Inference Time (s)": np.nan,
                    "FPS": np.nan,
                    "Latency (ms)": np.nan,
                    "Checkpoint Path": "",
                    "Export Format": "not_run",
                    "Export Path": "",
                    "Export Note": f"Run failed: {type(exc).__name__}: {exc}",
                }
            )

        partial_df = pd.DataFrame(comparison_results)
        partial_df.to_csv(REPORT_DIR / "yolo_autoaugment_model_comparison_partial.csv", index=False)
        partial_df.to_json(REPORT_DIR / "yolo_autoaugment_model_comparison_partial.json", orient="records", indent=2)


## 6. Combined Results

In [ ]:
df_summary = pd.DataFrame(comparison_results)
metric_columns = [
    "Augmentation",
    "Model",
    "Parameters (M)",
    "Training Time (s)",
    "Val F1-Score",
    "Test Accuracy",
    "Test F1-Score",
    "Cohen Kappa",
    "Inference Time (s)",
    "FPS",
    "Latency (ms)",
]

if not df_summary.empty:
    df_summary = df_summary.sort_values(by="Test F1-Score", ascending=False, na_position="last").reset_index(drop=True)
    print("\n" + "=" * 90)
    print("YOLO AUTO-AUGMENTATION MODEL PERFORMANCE SUMMARY: SHRIMP DISEASE CLASSIFICATION")
    print("=" * 90)
    display(df_summary[metric_columns])

    summary_csv = REPORT_DIR / "yolo_autoaugment_model_comparison_summary.csv"
    summary_json = REPORT_DIR / "yolo_autoaugment_model_comparison_summary.json"
    metrics_csv = REPORT_DIR / "yolo_autoaugment_model_comparison_metrics_table.csv"
    df_summary.to_csv(summary_csv, index=False)
    df_summary.to_json(summary_json, orient="records", indent=2)
    df_summary[metric_columns].to_csv(metrics_csv, index=False)
    print(f"Saved full summary CSV: {summary_csv}")
    print(f"Saved full summary JSON: {summary_json}")
    print(f"Saved metrics-only CSV: {metrics_csv}")

    best_by_aug = (
        df_summary.sort_values(by="Test F1-Score", ascending=False, na_position="last")
        .groupby("Augmentation", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    print("\nBest YOLO model per augmentation policy:")
    display(best_by_aug[metric_columns])
else:
    print("No model results were produced.")
